# Notebook 02 — Xác định thông số tốt nhất cho mô hình phân loại Good/Bad (Loan)

**Mục tiêu:** đánh giá dữ liệu hiện có, tạo đặc trưng từ việc ghép nhiều bảng, rồi dùng kết quả đó để **xác định baseline** và **cải tiến** (chọn feature set / cách xử lý mất cân bằng / mô hình / siêu tham số tốt nhất) cho bài toán dự đoán Good (0) / Bad (1).

**Nguyên tắc bất biến dữ liệu (tuyệt đối tuân thủ trong toàn bộ notebook):**

1. Không bao giờ gọi `.to_csv()` hoặc ghi đè lên file gốc trong `Data_cleaned`.
2. Không sửa cột gốc trên DataFrame đã đọc — mọi cột tính toán đều là cột **mới**, đặt tên riêng (ví dụ `date` gốc giữ nguyên, ngày đã parse lưu ở `..._date_parsed`).
3. Mọi bước join/feature engineering chỉ thao tác trên `.copy()` của dữ liệu gốc, không sửa trực tiếp dict `raw` chứa dữ liệu đã đọc.
4. Việc điền giá trị thiếu (imputation) **chỉ** xảy ra bên trong `Pipeline`/`ColumnTransformer` của scikit-learn lúc `fit_transform` để huấn luyện mô hình — không bao giờ `fillna()` ghi đè trực tiếp lên DataFrame nguồn hay lên tập train dùng để đánh giá. Hai loại NaN được phân biệt rõ: NaN "thực sự không biết" (ví dụ khoản vay chưa có giao dịch nào trước ngày vay → để Pipeline tự điền) và "0 cấu trúc" (ví dụ tài khoản không có lệnh thanh toán định kỳ nào → đúng là 0, không phải thiếu).
5. Cuối notebook có các đoạn `assert` kiểm chứng: dữ liệu gốc (`raw`) và tập train (`X_train_raw`) không hề bị thay đổi sau toàn bộ quá trình huấn luyện/đánh giá.

**Định nghĩa nhãn (đã thống nhất lại):** status **A, C → y=0 (Good)**; status **B, D → y=1 (Bad)**. Tỷ lệ Good/Bad sẽ được tính trực tiếp từ dữ liệu thật ở bước dưới, không dùng số giả định.


## 0. Kết nối Google Drive & cài thư viện

In [ ]:
try:
    from google.colab import drive
    drive.mount('/content/drive')
    # Sửa 'DSS_Project' thành tên thư mục bạn tạo trên Drive chứa dữ liệu
    BASE_DIR = '/content/drive/MyDrive/DSS_Project'
    print('Đã kết nối Google Drive thành công!')
except ImportError:
    BASE_DIR = '.'
    print('Không ở trong môi trường Colab, sử dụng đường dẫn local.')


In [ ]:
!pip install -q imbalanced-learn


In [ ]:
import os
import warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from IPython.display import display

from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.model_selection import (
    train_test_split, StratifiedKFold, cross_validate, cross_val_predict, RandomizedSearchCV
)
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier, HistGradientBoostingClassifier
from sklearn.metrics import (
    roc_auc_score, average_precision_score, recall_score, precision_score,
    f1_score, confusion_matrix
)

from imblearn.pipeline import Pipeline as ImbPipeline
from imblearn.over_sampling import SMOTE
from imblearn.under_sampling import RandomUnderSampler

warnings.filterwarnings('ignore')
sns.set_style('whitegrid')
pd.set_option('display.max_columns', 100)
pd.set_option('display.width', 150)

RANDOM_STATE = 42

DATA_DIR = os.path.join(BASE_DIR, 'Data_cleaned')
print(f'Thư mục dữ liệu: {DATA_DIR}')


## 1. Đọc dữ liệu (chỉ đọc — `raw` được xem là bất khả xâm phạm)

Chỉ đọc 8 bảng liên quan trực tiếp đến bài toán khoản vay (loan, account, client, disposition, district, card, order, trans). Dict `raw` này sẽ **không bao giờ** bị gán/sửa lại ở bất kỳ cell nào sau đây — mọi nơi cần dùng dữ liệu, ta gọi `raw[...].copy()`.

In [ ]:
TABLE_FILES = {
    'account':     'completedacct.csv',
    'client':      'completedclient.csv',
    'disposition': 'completeddisposition.csv',
    'district':    'completeddistrict.csv',
    'loan':        'completedloan.csv',
    'card':        'completedcard.csv',
    'order':       'completedorder.csv',
    'trans':       'completedtrans.csv',
}

raw = {}
for name, filename in TABLE_FILES.items():
    raw[name] = pd.read_csv(os.path.join(DATA_DIR, filename))
    print(f'  - {name:12s}: {raw[name].shape[0]:>9,} dòng x {raw[name].shape[1]:>2} cột')

# Lưu lại "dấu vân tay" của dữ liệu gốc để kiểm chứng bất biến ở cuối notebook
ORIG_SHAPES = {name: df.shape for name, df in raw.items()}
ORIG_NULL_COUNTS = {name: int(df.isnull().sum().sum()) for name, df in raw.items()}
print('\nĐã lưu snapshot (shape + tổng số null) của dữ liệu gốc để đối chiếu cuối notebook.')


## 2. Gán nhãn mục tiêu từ dữ liệu `status` THẬT (không dùng số giả định)

In [ ]:
status_counts = raw['loan']['status'].value_counts()
status_pct = (status_counts / status_counts.sum() * 100).round(2)
print('Phân phối status thực tế trong dữ liệu:')
display(pd.DataFrame({'count': status_counts, 'pct': status_pct}))

unexpected_status = set(raw['loan']['status'].unique()) - set(['A', 'B', 'C', 'D'])
if unexpected_status:
    print(f'CẢNH BÁO: có giá trị status ngoài A/B/C/D: {unexpected_status}')

STATUS_TO_TARGET = {'A': 0, 'C': 0, 'B': 1, 'D': 1}

loan_labeled = raw['loan'].copy()  # bản sao — không đụng tới raw['loan']
loan_labeled['target'] = loan_labeled['status'].map(STATUS_TO_TARGET)
loan_labeled['loan_date_parsed'] = pd.to_datetime(loan_labeled['date'], errors='coerce')

n_unmapped = loan_labeled['target'].isnull().sum()
if n_unmapped > 0:
    print(f'CẢNH BÁO: {n_unmapped} khoản vay có status không map được, sẽ bị loại khỏi tập huấn luyện.')

target_counts = loan_labeled['target'].value_counts(dropna=False)
target_pct = (target_counts / len(loan_labeled) * 100).round(2)
print('\nPhân phối nhãn mục tiêu thực tế (0=Good, 1=Bad):')
display(pd.DataFrame({'count': target_counts, 'pct': target_pct}))


## 3. Đặc trưng từ `account` + `district`

In [ ]:
account = raw['account'].copy()
account['account_open_date_parsed'] = pd.to_datetime(account['date'], errors='coerce')

district = raw['district'].copy()
district_small = district[['district_id', 'region', 'division']].rename(
    columns={'district_id': 'account_district_id'}
)

account_features = account[['account_id', 'district_id', 'frequency', 'account_open_date_parsed']].rename(
    columns={'district_id': 'account_district_id'}
)
account_features = account_features.merge(district_small, on='account_district_id', how='left')

print('account_features:', account_features.shape)
display(account_features.head(3))


## 4. Đặc trưng từ `client` + `disposition` (chủ tài khoản, số người sở hữu)

Lưu ý: `disposition.client_id` là `int64` còn `client.client_id` là `object` — ép kiểu trên **bản sao** để khớp, không sửa bảng gốc.

In [ ]:
disp = raw['disposition'].copy()
disp['client_id'] = disp['client_id'].astype(str)

client = raw['client'].copy()

# Số người có quyền trên tài khoản (chủ + người được ủy quyền)
n_disponents_df = disp.groupby('account_id').size().reset_index(name='n_disponents')

# Chủ tài khoản chính (OWNER)
owner_df = disp[disp['type'] == 'OWNER'][['account_id', 'client_id']]
dup_owner = owner_df['account_id'].duplicated().sum()
if dup_owner > 0:
    print(f'CẢNH BÁO: {dup_owner} account_id có nhiều hơn 1 OWNER — chỉ giữ dòng đầu tiên.')
    owner_df = owner_df.drop_duplicates(subset='account_id')

owner_client = owner_df.merge(
    client[['client_id', 'sex', 'age', 'district_id']], on='client_id', how='left'
).rename(columns={'sex': 'owner_sex', 'age': 'owner_age', 'district_id': 'owner_district_id'})

# is_local_client: chủ tài khoản có cùng khu vực với tài khoản hay không
owner_client = owner_client.merge(
    account[['account_id', 'district_id']].rename(columns={'district_id': 'account_district_id'}),
    on='account_id', how='left'
)
owner_client['is_local_client'] = (
    owner_client['owner_district_id'] == owner_client['account_district_id']
).astype(int)

client_features = owner_client[['account_id', 'owner_sex', 'owner_age', 'is_local_client']]

print('client_features:', client_features.shape)
display(client_features.head(3))
print('\nn_disponents_df:', n_disponents_df.shape)


## 5. Đặc trưng từ `card` (chỉ tính thẻ được cấp TRƯỚC ngày vay — tránh leakage)

In [ ]:
card = raw['card'].copy()
card['card_date_parsed'] = pd.to_datetime(card['date'], errors='coerce')

card_disp = card.merge(disp[['disp_id', 'account_id']], on='disp_id', how='left')
card_loan = card_disp.merge(
    loan_labeled[['account_id', 'loan_id', 'loan_date_parsed']], on='account_id', how='inner'
)
card_loan['has_card_flag'] = (card_loan['card_date_parsed'] < card_loan['loan_date_parsed']).astype(int)

has_card_df = card_loan.groupby('loan_id')['has_card_flag'].max().reset_index().rename(
    columns={'has_card_flag': 'has_card_before_loan'}
)

print('has_card_df:', has_card_df.shape, '| số khoản vay có thẻ trước ngày vay:', int(has_card_df['has_card_before_loan'].sum()))


## 6. Đặc trưng từ `order` (lệnh thanh toán định kỳ)

`completedorder.csv` **không có cột ngày** trong dữ liệu đã tóm tắt — đây được xem là cấu hình tĩnh của tài khoản (standing order), không gắn với một thời điểm cụ thể, nên không có rủi ro leakage thời gian như `trans`. Nếu sau khi kiểm tra schema thật trên Drive mà phát hiện có cột ngày, cần lọc lại theo `< loan_date_parsed` tương tự `trans`.

In [ ]:
order = raw['order'].copy()

has_order_date = any('date' in c.lower() for c in order.columns)
print(f'Bảng order có cột liên quan đến ngày? {has_order_date}')

order_features = order.groupby('account_id').agg(
    n_orders=('order_id', 'count'),
    total_order_amount=('amount', 'sum'),
    avg_order_amount=('amount', 'mean'),
).reset_index()

print('order_features:', order_features.shape)
display(order_features.head(3))


## 7. Đặc trưng từ `trans` — CHỈ lấy giao dịch trước ngày vay (chống leakage)

Đây là nhóm đặc trưng quan trọng nhất nhưng cũng rủi ro leakage cao nhất, vì `trans` chứa lịch sử giao dịch trải dài cả trước và sau ngày vay. Mọi thống kê đều được tính **chỉ** trên các giao dịch có `trans_date_parsed < loan_date_parsed` của đúng khoản vay đó.

In [ ]:
trans = raw['trans'].copy()
trans['trans_date_parsed'] = pd.to_datetime(trans['date'], errors='coerce')

n_parse_fail = trans['trans_date_parsed'].isnull().sum()
print(f'Số dòng trans không parse được ngày: {n_parse_fail} / {len(trans):,}')

# Join với loan (loan rất nhỏ, account_id ở trans rất lớn) -> chỉ giữ giao dịch của các account có khoản vay
trans_loan = trans.merge(
    loan_labeled[['account_id', 'loan_id', 'loan_date_parsed']], on='account_id', how='inner'
)
print('Số dòng trans thuộc các account có khoản vay:', len(trans_loan))

filtered = trans_loan[trans_loan['trans_date_parsed'] < trans_loan['loan_date_parsed']].copy()
print('Số dòng trans TRƯỚC ngày vay (sẽ dùng để tính đặc trưng):', len(filtered))

trans_agg = filtered.groupby('loan_id').agg(
    n_trans_before_loan=('trans_id', 'count'),
    avg_balance_before_loan=('balance', 'mean'),
    std_balance_before_loan=('balance', 'std'),
    min_balance_before_loan=('balance', 'min'),
    max_balance_before_loan=('balance', 'max'),
    avg_amount_before_loan=('amount', 'mean'),
    total_amount_before_loan=('amount', 'sum'),
).reset_index()

last_balance = (
    filtered.sort_values(['loan_id', 'trans_date_parsed'])
    .groupby('loan_id')
    .tail(1)[['loan_id', 'balance']]
    .rename(columns={'balance': 'last_balance_before_loan'})
)

trans_features = trans_agg.merge(last_balance, on='loan_id', how='left')

print('\ntrans_features:', trans_features.shape)
display(trans_features.head(3))


## 8. Ghép toàn bộ thành `df_model` (1 dòng = 1 khoản vay)

Quy tắc điền giá trị ngay tại bước ghép: **chỉ** fillna(0) cho các cột mà 0 là giá trị đúng về bản chất (đếm/tổng của tập rỗng — "không có" = 0 thật), **không** fillna cho các cột thống kê (trung bình/độ lệch chuẩn/số dư) khi nhóm rỗng — những cột đó để NaN thật, xử lý ở Pipeline lúc huấn luyện.

In [ ]:
df_model = loan_labeled[
    ['loan_id', 'account_id', 'amount', 'duration', 'payments', 'purpose', 'status', 'target', 'loan_date_parsed']
].copy()

df_model = df_model.merge(account_features, on='account_id', how='left')
df_model['account_age_days'] = (df_model['loan_date_parsed'] - df_model['account_open_date_parsed']).dt.days

df_model = df_model.merge(client_features, on='account_id', how='left')
df_model = df_model.merge(n_disponents_df, on='account_id', how='left')
df_model = df_model.merge(has_card_df, on='loan_id', how='left')
df_model = df_model.merge(order_features, on='account_id', how='left')
df_model = df_model.merge(trans_features, on='loan_id', how='left')

# "0 cấu trúc" — không có dòng nào ở bảng tương ứng nghĩa là đúng 0, không phải thiếu
ZERO_STRUCTURAL_COLS = [
    'n_disponents', 'has_card_before_loan', 'n_orders', 'total_order_amount',
    'avg_order_amount', 'n_trans_before_loan', 'total_amount_before_loan'
]
for col in ZERO_STRUCTURAL_COLS:
    df_model[col] = df_model[col].fillna(0)

# Loại các dòng không map được nhãn (nếu có)
df_model = df_model[df_model['target'].notnull()].copy()
df_model['target'] = df_model['target'].astype(int)

print('df_model:', df_model.shape)
display(df_model.head(3))


## 9. Đánh giá các thông số hiện tại trên `df_model` (EDA trên tập đã ghép)

Đây là bước "đánh giá các thông số hiện tại" theo đúng yêu cầu — kiểm tra null phát sinh từ việc ghép bảng (khác hẳn với dữ liệu gốc, nơi không có null), và xem feature nào có tín hiệu với nhãn trước khi đưa vào huấn luyện.

In [ ]:
def missing_report(df):
    n = len(df)
    miss = df.isnull().sum()
    pct = (miss / n * 100).round(2)
    rep = pd.DataFrame({'non_null': n - miss, 'null_count': miss, 'null_pct': pct, 'dtype': df.dtypes})
    return rep.sort_values('null_count', ascending=False)

rep = missing_report(df_model)
print('Các cột có giá trị thiếu sau khi ghép bảng (giải thích nguyên nhân, KHÔNG sửa df_model ở đây):')
display(rep[rep['null_count'] > 0])
print('\n=> Các giá trị thiếu này chủ yếu do khoản vay được cấp ngay sau khi mở tài khoản,')
print('   nên không có giao dịch nào trước ngày vay -> các thống kê balance/amount không xác định.')
print('   Việc điền giá trị sẽ được thực hiện bên trong Pipeline ở bước huấn luyện, KHÔNG sửa df_model.')


In [ ]:
numeric_cols_check = df_model.select_dtypes(include=[np.number]).drop(columns=['target', 'loan_id'], errors='ignore')
corr_with_target = numeric_cols_check.corrwith(df_model['target']).sort_values(key=abs, ascending=False)
print('Tương quan (Pearson) giữa các biến số và nhãn target (tham khảo sơ bộ, không phải bằng chứng nhân quả):')
display(corr_with_target.to_frame('corr_with_target'))


## 10. Định nghĩa các tập đặc trưng (feature set) để so sánh

In [ ]:
FEATURE_SETS = {
    'loan_only': (
        ['amount', 'duration', 'payments'],
        ['purpose']
    ),
    'loan_account_district_client': (
        ['amount', 'duration', 'payments', 'account_age_days', 'owner_age', 'n_disponents'],
        ['purpose', 'frequency', 'region', 'division', 'owner_sex', 'is_local_client', 'has_card_before_loan']
    ),
    'full_with_trans': (
        ['amount', 'duration', 'payments', 'account_age_days', 'owner_age', 'n_disponents',
         'n_orders', 'total_order_amount', 'avg_order_amount',
         'n_trans_before_loan', 'avg_balance_before_loan', 'std_balance_before_loan',
         'min_balance_before_loan', 'max_balance_before_loan', 'last_balance_before_loan',
         'avg_amount_before_loan', 'total_amount_before_loan'],
        ['purpose', 'frequency', 'region', 'division', 'owner_sex', 'is_local_client', 'has_card_before_loan']
    ),
}

for name, (num_cols, cat_cols) in FEATURE_SETS.items():
    print(f'{name:30s}: {len(num_cols)} biến số + {len(cat_cols)} biến phân loại')


## 11. Tách train/test (Stratified) — giữ nguyên NaN, không sửa bất cứ giá trị nào

In [ ]:
ALL_FEATURE_COLS = sorted(set(
    c for cols_pair in FEATURE_SETS.values() for c in (cols_pair[0] + cols_pair[1])
))

X_all_raw = df_model[ALL_FEATURE_COLS].copy()
y_all = df_model['target'].copy()

X_train_raw, X_test_raw, y_train, y_test = train_test_split(
    X_all_raw, y_all, test_size=0.2, stratify=y_all, random_state=RANDOM_STATE
)

print(f'Train: {X_train_raw.shape}  | Bad trong train: {int(y_train.sum())} / {len(y_train)}')
print(f'Test:  {X_test_raw.shape}  | Bad trong test:  {int(y_test.sum())} / {len(y_test)}')

TRAIN_NULL_BEFORE = int(X_train_raw.isnull().sum().sum())
TEST_NULL_BEFORE = int(X_test_raw.isnull().sum().sum())
print(f'\nSố lượng null trong X_train_raw (sẽ kiểm chứng lại cuối notebook): {TRAIN_NULL_BEFORE}')


## 12. Xây pipeline tiền xử lý — null CHỈ được xử lý trong Pipeline, không đụng DataFrame nguồn

Hai phiên bản: bản chuẩn (Imputer + Scaler/OneHot, dùng cho Logistic Regression / Random Forest) và bản "native NaN" (không impute biến số, để `HistGradientBoostingClassifier` tự xử lý missing values theo cơ chế nội tại của nó — một cách xử lý null hoàn toàn khác, không cần điền giá trị).

In [ ]:
def create_preprocessor(numeric_features, categorical_features, scale_numeric=True, numeric_strategy='mean'):
    """Pipeline tiêu chuẩn: Imputer + (Scaler) cho biến số, Imputer + OneHot cho biến phân loại."""
    numeric_steps = [('imputer', SimpleImputer(strategy=numeric_strategy))]
    if scale_numeric:
        numeric_steps.append(('scaler', StandardScaler()))
    numeric_transformer = Pipeline(steps=numeric_steps)

    categorical_transformer = Pipeline(steps=[
        ('imputer', SimpleImputer(strategy='most_frequent')),
        ('onehot', OneHotEncoder(handle_unknown='ignore'))
    ])

    return ColumnTransformer(transformers=[
        ('num', numeric_transformer, numeric_features),
        ('cat', categorical_transformer, categorical_features)
    ], remainder='drop')


def create_preprocessor_native_nan(numeric_features, categorical_features):
    """Biến số giữ nguyên NaN (passthrough) để HistGradientBoosting tự xử lý; biến phân loại vẫn cần impute + onehot."""
    categorical_transformer = Pipeline(steps=[
        ('imputer', SimpleImputer(strategy='most_frequent')),
        ('onehot', OneHotEncoder(handle_unknown='ignore'))
    ])
    return ColumnTransformer(transformers=[
        ('num', 'passthrough', numeric_features),
        ('cat', categorical_transformer, categorical_features)
    ], remainder='drop')

print('Đã định nghĩa create_preprocessor() và create_preprocessor_native_nan().')


## 13. XÁC ĐỊNH BASELINE — lưới so sánh feature set × cách xử lý mất cân bằng × mô hình

Dùng `imblearn.Pipeline` (không phải `sklearn.Pipeline` thường) để SMOTE/undersampling chỉ được áp dụng trên phần train của từng fold trong `StratifiedKFold` — tránh rò rỉ thông tin do oversample trước khi chia fold. Vì `HistGradientBoostingClassifier` không có tham số `class_weight`, tổ hợp (hgb, class_weight) bị bỏ qua.

In [ ]:
def build_pipeline(model_name, feature_set_name, imbalance_method):
    numeric_features, categorical_features = FEATURE_SETS[feature_set_name]

    if model_name == 'hgb':
        if imbalance_method == 'class_weight':
            return None
        preprocessor = create_preprocessor_native_nan(numeric_features, categorical_features)
        model = HistGradientBoostingClassifier(random_state=RANDOM_STATE)
    elif model_name == 'logreg':
        preprocessor = create_preprocessor(numeric_features, categorical_features, scale_numeric=True)
        class_weight = 'balanced' if imbalance_method == 'class_weight' else None
        model = LogisticRegression(max_iter=2000, class_weight=class_weight, random_state=RANDOM_STATE)
    elif model_name == 'random_forest':
        preprocessor = create_preprocessor(numeric_features, categorical_features, scale_numeric=False)
        class_weight = 'balanced' if imbalance_method == 'class_weight' else None
        model = RandomForestClassifier(n_estimators=300, class_weight=class_weight, random_state=RANDOM_STATE)
    else:
        raise ValueError(f'Model không hợp lệ: {model_name}')

    steps = [('preprocessor', preprocessor)]
    if imbalance_method == 'smote':
        steps.append(('sampler', SMOTE(random_state=RANDOM_STATE, k_neighbors=3)))
    elif imbalance_method == 'undersample':
        steps.append(('sampler', RandomUnderSampler(random_state=RANDOM_STATE)))
    steps.append(('clf', model))
    return ImbPipeline(steps=steps)

print('Đã định nghĩa build_pipeline().')


In [ ]:
skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)
scoring = {
    'roc_auc': 'roc_auc',
    'pr_auc': 'average_precision',
    'recall': 'recall',
    'precision': 'precision',
    'f1': 'f1',
}

baseline_rows = []
for feature_set_name in FEATURE_SETS:
    for model_name in ['logreg', 'random_forest', 'hgb']:
        for imbalance_method in ['none', 'class_weight', 'smote', 'undersample']:
            pipe = build_pipeline(model_name, feature_set_name, imbalance_method)
            if pipe is None:
                continue
            try:
                cv_res = cross_validate(
                    pipe, X_train_raw, y_train, cv=skf, scoring=scoring, n_jobs=-1, error_score='raise'
                )
                baseline_rows.append({
                    'feature_set': feature_set_name,
                    'model': model_name,
                    'imbalance_method': imbalance_method,
                    'roc_auc': cv_res['test_roc_auc'].mean(),
                    'pr_auc': cv_res['test_pr_auc'].mean(),
                    'recall': cv_res['test_recall'].mean(),
                    'precision': cv_res['test_precision'].mean(),
                    'f1': cv_res['test_f1'].mean(),
                })
            except Exception as e:
                print(f'Bỏ qua ({feature_set_name}, {model_name}, {imbalance_method}) do lỗi: {e}')

In [ ]:
if not baseline_rows:
    raise ValueError('Không có tổ hợp baseline nào chạy thành công.')
baseline_results = pd.DataFrame(baseline_rows).sort_values(
    ['pr_auc', 'f1'], ascending=False
).reset_index(drop=True)
best_row = baseline_results.iloc[0]
print(f'Tổng số tổ hợp chạy thành công: {len(baseline_results)}')
display(baseline_results)


## 14. CẢI TIẾN — dò siêu tham số cho tổ hợp baseline tốt nhất

### Columns in `df_model`

In [ ]:
print(df_model.columns.tolist())

In [ ]:
PARAM_DISTRIBUTIONS = {
    'logreg': {
        'clf__C': [0.001, 0.01, 0.1, 1, 10, 100],
        'clf__penalty': ['l2'],
    },
    'random_forest': {
        'clf__n_estimators': [100, 200, 300, 500],
        'clf__max_depth': [None, 5, 10, 20],
        'clf__min_samples_leaf': [1, 2, 5, 10],
        'clf__max_features': ['sqrt', 'log2', None],
    },
    'hgb': {
        'clf__max_iter': [100, 200, 300, 500],
        'clf__max_depth': [None, 3, 5, 8],
        'clf__learning_rate': [0.01, 0.05, 0.1, 0.2],
        'clf__l2_regularization': [0.0, 0.1, 1.0],
    },
}

best_model_name = best_row['model']
best_feature_set_name = best_row['feature_set']
best_imbalance_method = best_row['imbalance_method']

base_pipeline = build_pipeline(best_model_name, best_feature_set_name, best_imbalance_method)

search = RandomizedSearchCV(
    base_pipeline,
    param_distributions=PARAM_DISTRIBUTIONS[best_model_name],
    n_iter=25,
    scoring='average_precision',
    cv=skf,
    random_state=RANDOM_STATE,
    n_jobs=-1,
    refit=True,
)
search.fit(X_train_raw, y_train)

print('Siêu tham số tốt nhất:', search.best_params_)
print(f'PR-AUC sau khi dò siêu tham số: {search.best_score_:.4f}')
print(f"PR-AUC baseline (trước dò): {best_row['pr_auc']:.4f}")
print(f"Cải thiện: {search.best_score_ - best_row['pr_auc']:+.4f}")

final_model = search.best_estimator_


## 15. Đánh giá mô hình cuối trên tập test giữ riêng (hold-out)

In [ ]:
y_test_proba = final_model.predict_proba(X_test_raw)[:, 1]
y_test_pred = (y_test_proba >= 0.5).astype(int)

test_roc_auc = roc_auc_score(y_test, y_test_proba)
test_pr_auc = average_precision_score(y_test, y_test_proba)
test_recall = recall_score(y_test, y_test_pred)
test_precision = precision_score(y_test, y_test_pred)
test_f1 = f1_score(y_test, y_test_pred)
test_cm = confusion_matrix(y_test, y_test_pred)

print('KẾT QUẢ TRÊN TẬP TEST (hold-out, mô hình chưa từng thấy):')
print(f'  ROC-AUC   : {test_roc_auc:.4f}')
print(f'  PR-AUC    : {test_pr_auc:.4f}')
print(f'  Recall(Bad)    : {test_recall:.4f}')
print(f'  Precision(Bad) : {test_precision:.4f}')
print(f'  F1(Bad)        : {test_f1:.4f}')
print('\nConfusion matrix (hàng=thực tế, cột=dự đoán, [Good, Bad]):')
display(pd.DataFrame(test_cm, index=['Thực: Good', 'Thực: Bad'], columns=['Dự đoán: Good', 'Dự đoán: Bad']))

## 16. Mô phỏng phân luồng quyết định DSS (ngưỡng tính từ dữ liệu thật, không giả định)

Ngưỡng được tính trên **dự đoán out-of-fold của tập train** (qua `cross_val_predict`) — không dùng tập test để chọn ngưỡng, tránh rò rỉ thông tin quyết định. Các mốc phần trăm (20%/90%) chỉ là điểm khởi đầu minh họa, có thể điều chỉnh theo khả năng vận hành thực tế của ngân hàng.

In [ ]:
AUTO_APPROVE_PCT = 20   # 20% hồ sơ rủi ro thấp nhất -> tự động duyệt
AUTO_REJECT_PCT = 90    # 10% hồ sơ rủi ro cao nhất -> tự động từ chối

train_oof_proba = cross_val_predict(
    final_model, X_train_raw, y_train, cv=skf, method='predict_proba', n_jobs=-1
)[:, 1]

low_thr = np.percentile(train_oof_proba, AUTO_APPROVE_PCT)
high_thr = np.percentile(train_oof_proba, AUTO_REJECT_PCT)
print(f'Ngưỡng Auto-Approve (score <): {low_thr:.4f}')
print(f'Ngưỡng Auto-Reject  (score >): {high_thr:.4f}')

def route(score):
    if score < low_thr:
        return 'Auto-Approve'
    elif score > high_thr:
        return 'Auto-Reject'
    return 'Manual-Review'

test_routes = pd.Series(y_test_proba, index=y_test.index).apply(route)
routing_summary = pd.DataFrame({'route': test_routes, 'actual_bad': y_test}).groupby('route')['actual_bad'].agg(
    n_ho_so='count', bad_rate='mean'
)
print('\nKết quả phân luồng trên tập test (số liệu thật):')
display(routing_summary)


## 17. Kiểm chứng bất biến dữ liệu (Immutability check)

In [ ]:
all_ok = True
for name, df in raw.items():
    if df.shape != ORIG_SHAPES[name]:
        print(f'LỖI: bảng {name} đã đổi shape!')
        all_ok = False
    if int(df.isnull().sum().sum()) != ORIG_NULL_COUNTS[name]:
        print(f'LỖI: bảng {name} đã đổi số lượng null!')
        all_ok = False

train_null_after = int(X_train_raw.isnull().sum().sum())
test_null_after = int(X_test_raw.isnull().sum().sum())
if train_null_after != TRAIN_NULL_BEFORE:
    print('LỖI: X_train_raw đã bị thay đổi (số null khác trước)!')
    all_ok = False
if test_null_after != TEST_NULL_BEFORE:
    print('LỖI: X_test_raw đã bị thay đổi (số null khác trước)!')
    all_ok = False

if all_ok:
    print('✔ Toàn bộ dữ liệu gốc (raw) và tập train/test (X_train_raw, X_test_raw) KHÔNG bị thay đổi')
    print('  trong suốt quá trình ghép bảng, tạo đặc trưng, huấn luyện và đánh giá mô hình.')


## 18. Tổng kết — "Thông số tốt nhất" xác định được

In [ ]:
print('=' * 70)
print('TỔNG KẾT THÔNG SỐ TỐT NHẤT')
print('=' * 70)
print(f'Feature set tốt nhất       : {best_feature_set_name}')
print(f'Cách xử lý mất cân bằng    : {best_imbalance_method}')
print(f'Mô hình                    : {best_model_name}')
print(f'Siêu tham số tốt nhất      : {search.best_params_}')
print(f'PR-AUC (CV, sau dò tham số): {search.best_score_:.4f}')
print(f'PR-AUC (test, hold-out)    : {test_pr_auc:.4f}')
print(f'ROC-AUC (test, hold-out)   : {test_roc_auc:.4f}')
print(f'Recall/Precision/F1 (Bad)  : {test_recall:.4f} / {test_precision:.4f} / {test_f1:.4f}')
print('=' * 70)
print('Toàn bộ kết quả trên dựa hoàn toàn vào dữ liệu thật đọc từ Data_cleaned,')
print('không dùng số liệu giả định nào, và dữ liệu gốc không bị thay đổi (xem mục 17).')
